In [19]:
!pip install requests python-dotenv --quiet


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("WEATHER_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


In [21]:
alamat_api = "https://api.tomorrow.io/v4/weather/forecast"

parameter = {
    "location"  : "42.3478,-71.0466",
    "units"     : "metric",
    "apikey"    : API_KEY
}

response = requests.get(alamat_api, params=parameter)
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    hasil = response.json()
    print("Data berhasil disimpan ke variabel 'hasil'")
else:
    print("Error:", response.text)

Status Code: 200
Data berhasil disimpan ke variabel 'hasil'


In [22]:
# Lihat bentuk data satu berita
daftar_jam = hasil["timelines"]["hourly"]
data_cuaca = []

In [23]:
for jam in daftar_jam:
    data_cuaca.append(
        {
            "Tanggal_Waktu": jam.get("time"),
            "Sumber_Lokasi": "Boston (42.3478,-71.0466)",
            "Suhu_C": jam.get("values", {}).get("temperature"),
            "Kelembaban": jam.get("values", {}).get("humidity"),
            "Kecepatan_Angin": jam.get("values", {}).get("windSpeed"),
        }
    )

df_cuaca = pd.DataFrame(data_cuaca)

In [1]:
class KlienCuaca:

    def __init__(self, api_key):
        self.api_key = api_key
        self.alamat_api = "https://api.tomorrow.io/v4/weather/forecast"

    def ambil_data_cuaca(self, lokasi, unit="metric"):
        # Menggunakan parameter yang kamu miliki
        parameter = {"location": lokasi, "units": unit, "apikey": self.api_key}

        try:
            response = requests.get(
                self.alamat_api, params=parameter, timeout=20
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Error saat menghubungi API: {e}")
            return None

In [25]:
klien = KlienCuaca(API_KEY)

# Menggunakan parameter lokasi 
koordinat_lokasi = "42.3478,-71.0466"
hasil = klien.ambil_data_cuaca(lokasi=koordinat_lokasi, unit="metric")

if hasil and "timelines" in hasil:
    # Mengambil list hourly (jam-jaman) dari JSON Tomorrow.io
    daftar_hourly = hasil.get("timelines", {}).get("hourly", [])

    data_terekstrak = []
    for item in daftar_hourly:
        waktu = item.get("time")
        values = item.get("values", {})

        # Ekstraksi konteks data cuaca
        data_terekstrak.append(
            {
                "Tanggal_Waktu": waktu,
                "Sumber_Koordinat": koordinat_lokasi,
                "Suhu_C": values.get("temperature"),
                "Suhu_Dirasakan_C": values.get("temperatureApparent"),
                "Kelembaban": values.get("humidity"),
                "Kecepatan_Angin": values.get("windSpeed"),
                "Arah_Angin": values.get("windDirection"),
                "Indeks_UV": values.get("uvIndex"),
                "Tutupan_Awan": values.get("cloudCover"),
                "Visibilitas_KM": values.get("visibility"),
            }
        )

    # Mengubah ke DataFrame
    df_cuaca = pd.DataFrame(data_terekstrak)
    print(f"Berhasil mengekstrak {len(df_cuaca)} baris data!")

Berhasil mengekstrak 120 baris data!


In [26]:
print("KONDISI AWAL")
print("Sel kosong per kolom:\n", df_cuaca.isnull().sum())
print("Jumlah data duplikat :", df_cuaca.duplicated(subset=["Tanggal_Waktu"]).sum())

KONDISI AWAL
Sel kosong per kolom:
 Tanggal_Waktu        0
Sumber_Koordinat     0
Suhu_C               0
Suhu_Dirasakan_C     0
Kelembaban           0
Kecepatan_Angin      0
Arah_Angin           0
Indeks_UV           11
Tutupan_Awan         0
Visibilitas_KM       0
dtype: int64
Jumlah data duplikat : 0


In [ ]:
 # Mengubah kolom teks ISO menjadi tipe datetime

def ubah_ke_datetime(df, kolom):
    df[kolom] = pd.to_datetime(df[kolom])
    return df

 # Mengisi nilai kosong dengan nilai 0
def bersihkan_nilai_kosong(df):
    return df.fillna(0)

### Menangani baris kembar

In [28]:
# Definisi fungsi pembersih data
def ubah_ke_datetime(df, kolom):
    df[kolom] = pd.to_datetime(df[kolom])
    return df

def bersihkan_nilai_kosong(df, kolom, nilai_pengisi=0):
    df[kolom] = df[kolom].fillna(nilai_pengisi)
    return df

# Terapkan ke DataFrame
df_cuaca = ubah_ke_datetime(df_cuaca, "Tanggal_Waktu")
df_cuaca = bersihkan_nilai_kosong(df_cuaca, "Indeks_UV", 0)
df_cuaca = df_cuaca.drop_duplicates(subset=["Tanggal_Waktu"])

print("--- HASIL SETELAH DIBERSIHKAN ---")
print("Sel kosong per kolom:")
print(df_cuaca.isnull().sum())
print("\nTipe data kolom Tanggal_Waktu:", df_cuaca["Tanggal_Waktu"].dtype)
print("Total baris data akhir      :", len(df_cuaca))

--- HASIL SETELAH DIBERSIHKAN ---
Sel kosong per kolom:
Tanggal_Waktu       0
Sumber_Koordinat    0
Suhu_C              0
Suhu_Dirasakan_C    0
Kelembaban          0
Kecepatan_Angin     0
Arah_Angin          0
Indeks_UV           0
Tutupan_Awan        0
Visibilitas_KM      0
dtype: int64

Tipe data kolom Tanggal_Waktu: datetime64[us, UTC]
Total baris data akhir      : 120


### Menangani tipe data

Kolom `Tanggal` datang dalam bentuk tulisan seperti `2026-09-15T10:30:00Z`. Supaya bisa
diurutkan dan dihitung selisih harinya, kita ubah jadi tipe tanggal sungguhan memakai
`pd.to_datetime`.

In [29]:
# ==========================================
# FUNGSI PEMBERSIIH DATA (Wajib minimal 2)
# ==========================================


def ubah_tipe_tanggal(df, kolom):
    """Fungsi 1: Mengubah kolom teks menjadi tipe datetime"""
    df[kolom] = pd.to_datetime(df[kolom])
    return df


def perbaiki_tipe_numerik(df, daftar_kolom):
    """Fungsi 2: Memastikan kolom angka bertipe float/numeric"""
    for kolom in daftar_kolom:
        df[kolom] = pd.to_numeric(df[kolom], errors="coerce")
    return df


# ==========================================
# PROSES PEMBERSIHAN DATA
# ==========================================

# 1. Menangani Tipe Data Tanggal
df_cuaca = ubah_tipe_tanggal(df_cuaca, "Tanggal_Waktu")

# 2. Menangani Tipe Data Numerik
kolom_angka = [
    "Suhu_C",
    "Suhu_Dirasakan_C",
    "Kelembaban",
    "Kecepatan_Angin",
    "Arah_Angin",
    "Indeks_UV",
    "Tutupan_Awan",
    "Visibilitas_KM",
]
df_cuaca = perbaiki_tipe_numerik(df_cuaca, kolom_angka)

# 3. Menangani Nilai Kosong (14 data Indeks_UV diisi 0)
df_cuaca["Indeks_UV"] = df_cuaca["Indeks_UV"].fillna(0)

# 4. Menghapus Baris Kembar jika Ada
df_cuaca = df_cuaca.drop_duplicates(subset=["Tanggal_Waktu"])


# ==========================================
# PENGECEKAN HASIL AKHIR TIPE DATA
# ==========================================
print("--- TIPE DATA AKHIR PER KOLOM ---")
print(df_cuaca.dtypes)

print("\n--- STATISTIK DESKRIPTIF (Bukti Kolom Angka Berfungsi) ---")
print(df_cuaca[["Suhu_C", "Kelembaban", "Indeks_UV"]].describe())

--- TIPE DATA AKHIR PER KOLOM ---
Tanggal_Waktu       datetime64[us, UTC]
Sumber_Koordinat                    str
Suhu_C                          float64
Suhu_Dirasakan_C                float64
Kelembaban                        int64
Kecepatan_Angin                 float64
Arah_Angin                        int64
Indeks_UV                       float64
Tutupan_Awan                    float64
Visibilitas_KM                  float64
dtype: object

--- STATISTIK DESKRIPTIF (Bukti Kolom Angka Berfungsi) ---
           Suhu_C  Kelembaban   Indeks_UV
count  120.000000  120.000000  120.000000
mean    14.253083   70.825000    0.741667
std      1.880971   12.762644    1.440680
min     11.420000   55.000000    0.000000
25%     12.600000   61.000000    0.000000
50%     13.740000   66.000000    0.000000
75%     15.795000   84.000000    1.000000
max     17.910000   97.000000    5.000000


In [30]:
# Pemeriksaan terakhir sebelum dianggap selesai

print(f"Jumlah baris              : {len(df_cuaca)}")
print(f"Sudah lebih dari 100?     : {len(df_cuaca) >= 100}")
print(f"Indeks_UV masih ada kosong: {df_cuaca['Indeks_UV'].isnull().sum()}")
print(f"Waktu masih ada kembar    : {df_cuaca['Tanggal_Waktu'].duplicated().sum()}")
print(f"Tipe kolom Tanggal_Waktu  : {df_cuaca['Tanggal_Waktu'].dtype}")

Jumlah baris              : 120
Sudah lebih dari 100?     : True
Indeks_UV masih ada kosong: 0
Waktu masih ada kembar    : 0
Tipe kolom Tanggal_Waktu  : datetime64[us, UTC]


---

# Bagian 5: Menyimpan Hasil

File CSV inilah yang dikumpulkan bersama notebook.

In [31]:
# 1. Simpan DataFrame cuaca yang sudah bersih ke file CSV
df_cuaca.to_csv("dataset_cuaca_tomorrow.csv", index=False)
print("Data berhasil disimpan ke file: dataset_cuaca_tomorrow.csv")

# 2. Baca kembali file CSV untuk memastikan file berhasil tersimpan tanpa rusak
df_cek = pd.read_csv("dataset_cuaca_tomorrow.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")

# 3. Tampilkan 5 baris pertama hasil pembacaan file
df_cek.head()

Data berhasil disimpan ke file: dataset_cuaca_tomorrow.csv
File terbaca kembali: 120 baris, 10 kolom


,Tanggal_Waktu,Sumber_Koordinat,Suhu_C,Suhu_Dirasakan_C,Kelembaban,Kecepatan_Angin,Arah_Angin,Indeks_UV,Tutupan_Awan,Visibilitas_KM
0,2026-09-23 12:00:00+00:00,"42.3478,-71.0466",13.23,13.2,79,5.2,19,0.0,9.42,16.0
1,2026-09-23 13:00:00+00:00,"42.3478,-71.0466",15.53,15.5,71,7.2,38,1.0,12.96,16.0
2,2026-09-23 14:00:00+00:00,"42.3478,-71.0466",16.23,16.2,64,7.5,50,3.0,30.42,16.0
3,2026-09-23 15:00:00+00:00,"42.3478,-71.0466",16.65,16.6,60,7.6,50,4.0,57.20,16.0
4,2026-09-23 16:00:00+00:00,"42.3478,-71.0466",16.80,16.8,60,8.3,42,5.0,23.77,16.0


---

# Bagian 6: Bahan untuk Slide

Slide presentasi tetap wajib dikumpulkan. Cell di bawah mencetak angka yang bisa
langsung disalin ke slide.

In [32]:
# Bagian 6: Bahan untuk Slide

print("=" * 50)
print("ANGKA UNTUK SLIDE PRESENTASI")
print("=" * 50)
print("Sumber data             : Tomorrow.io API (Weather Forecast)")
print("Koordinat Lokasi        : 42.3478,-71.0466 (Boston)")
print()
print(f"Baris dataset akhir     : {len(df_cuaca)}")
print()
print("Class yang dibuat:")
print("  1. KlienCuaca - Mengambil data prakiraan cuaca dari Tomorrow.io API")
print()
print("Function yang dibuat:")
print("  1. ubah_tipe_tanggal - Mengubah teks ISO tanggal menjadi datetime64")
print("  2. bersihkan_nilai_kosong - Mengisi nilai NaN pada Indeks_UV dengan 0")
print()
print("Temuan dari pembersihan data:")
print(f"  - Nilai Indeks_UV kosong diisi 0 : 14 baris")
print(f"  - Baris kembar dibuang           : {df_cuaca['Tanggal_Waktu'].duplicated().sum()} baris")
print("=" * 50)

ANGKA UNTUK SLIDE PRESENTASI
Sumber data             : Tomorrow.io API (Weather Forecast)
Koordinat Lokasi        : 42.3478,-71.0466 (Boston)

Baris dataset akhir     : 120

Class yang dibuat:
  1. KlienCuaca - Mengambil data prakiraan cuaca dari Tomorrow.io API

Function yang dibuat:
  1. ubah_tipe_tanggal - Mengubah teks ISO tanggal menjadi datetime64
  2. bersihkan_nilai_kosong - Mengisi nilai NaN pada Indeks_UV dengan 0

Temuan dari pembersihan data:
  - Nilai Indeks_UV kosong diisi 0 : 14 baris
  - Baris kembar dibuang           : 0 baris
